In [43]:
import pypsa
import pandas as pd
import numpy as np

Parse the dataset (household)

In [44]:
csv_data = pd.read_csv("household_data_60min_singleindex.csv",
                       parse_dates=["utc_timestamp"],
                       index_col="utc_timestamp",
                       usecols=lambda col: "utc_timestamp" in col or ("residential" in col and ("grid_import" in col or "pv" in col)) or "residential" in col
                       )
print(csv_data.info())


<class 'pandas.DataFrame'>
DatetimeIndex: 38454 entries, 2014-12-11 17:00:00+00:00 to 2019-05-01 22:00:00+00:00
Data columns (total 39 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   DE_KN_residential1_dishwasher        15845 non-null  float64
 1   DE_KN_residential1_freezer           12896 non-null  float64
 2   DE_KN_residential1_grid_import       15873 non-null  float64
 3   DE_KN_residential1_heat_pump         15873 non-null  float64
 4   DE_KN_residential1_pv                15873 non-null  float64
 5   DE_KN_residential1_washing_machine   15846 non-null  float64
 6   DE_KN_residential2_circulation_pump  19823 non-null  float64
 7   DE_KN_residential2_dishwasher        19134 non-null  float64
 8   DE_KN_residential2_freezer           10251 non-null  float64
 9   DE_KN_residential2_grid_import       15798 non-null  float64
 10  DE_KN_residential2_washing_machine   24733 non-null  float

In [45]:
start_time = "2016-07-15 00:00:00"
end_time = "2016-07-15 23:00:00"
dataset_day = csv_data.loc[start_time:end_time]
print(dataset_day)


load_cols = [col for col in dataset_day.columns if "grid_import" in col]
pv_cols = [col for col in dataset_day.columns if "pv" in col ]

if (len(load_cols) == 0 or len(pv_cols) == 0):
    raise ValueError("Could not find consumption or solar columns")

print(f"Found {len(load_cols)} loads and {len(pv_cols)} pv")

                           DE_KN_residential1_dishwasher  \
utc_timestamp                                              
2016-07-15 00:00:00+00:00                        189.028   
2016-07-15 01:00:00+00:00                        189.028   
2016-07-15 02:00:00+00:00                        189.028   
2016-07-15 03:00:00+00:00                        189.028   
2016-07-15 04:00:00+00:00                        189.028   
2016-07-15 05:00:00+00:00                        189.028   
2016-07-15 06:00:00+00:00                        189.028   
2016-07-15 07:00:00+00:00                        189.028   
2016-07-15 08:00:00+00:00                        189.028   
2016-07-15 09:00:00+00:00                        189.028   
2016-07-15 10:00:00+00:00                        189.028   
2016-07-15 11:00:00+00:00                        189.028   
2016-07-15 12:00:00+00:00                        189.028   
2016-07-15 13:00:00+00:00                        189.028   
2016-07-15 14:00:00+00:00               

Create a PyPSA network and set the snapshots for a period of 24 h.

In [46]:
network = pypsa.Network()
timestamps = pd.date_range(start_time, periods=24, freq="h")
network.set_snapshots(timestamps)

Add the buses for the IEEE 33-bus system. The base voltage is 12.66 kV

In [47]:
for i in range(1, 34):
    network.add(
        "Bus",
        f"Bus_{i}",
        v_nom=12.66
    )

Adding the lines. Resistance (r) and reactance (x) are in Ohm. The thermal capacity (s_nom) is set high so as to not be a limiting factor

In [48]:
# (from, to, r, x)
line_data = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (from_b, to_b, r, x) in enumerate(line_data):
    network.add(
        "Line", 
        f"Line_{from_b}-{to_b}",
        bus0=f"Bus_{from_b}",
        bus1=f"Bus_{to_b}",
        r=r,
        x=x,
        s_nom=5000
    )

Add the loads. p_set is the active power (kW) and q_set is the reactive power (kVAr)

In [49]:
load_data = [
    (100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]

Define all the profiles

In [50]:
# raw_load_profile = np.sin(np.linspace(0, 2 * np.pi, 24)) * 0.4 + 0.6
price_profile = np.array([50, 45, 40, 40, 42, 48, 60, 75, 70, 65, 50, 30,
                          20, 20, 25, 45, 65, 80, 90, 100, 85, 70, 60, 55])
grid_price_series = pd.Series(price_profile, index=network.snapshots)

# raw_pv_profile = np.sin(np.linspace(-np.pi / 2, 3 * np.pi / 2, 24))
# raw_pv_profile[raw_pv_profile < 0] = 0
# pv_series = pd.Series(raw_pv_profile, index=network.snapshots)


Apply the time varying profiles

In [51]:
for i, (p, q) in enumerate(load_data):

    raw_load_col_name = load_cols[i % len(load_cols)]
    raw_load = dataset_day[raw_load_col_name]
    raw_load_series = raw_load.values
    normalized_load = raw_load_series / raw_load_series.max()
    real_load_profile = normalized_load * p

    # load_profile_series = pd.Series(p * raw_load_profile, index=network.snapshots)
    network.add(
        "Load",
        f"Load_bus_{i+2}",
        bus=f"Bus_{i+2}",
        p_set=real_load_profile,
        q_set=q
    )

Add grid, PV and batteries

In [54]:
network.add(
    "Generator",
    "Substation",
    bus="Bus_1",
    p_nom=10000,
    p_min_pu=-1.0,
    carrier="gas",
    marginal_cost=grid_price_series
)

for i in range(2, 34):

    raw_pv_col = pv_cols[i % len(pv_cols)]
    raw_pv = dataset_day[raw_pv_col].fillna(0)
    raw_pv_series = raw_pv.values
    real_pv_profile = raw_pv_series / raw_pv_series.max()
    # real_pv_profile = real_pv_profile.fillna(0)

    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=150,
        p_max_pu=real_pv_profile,
        carrier="solar",
        marginal_cost=0
    )
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=200, # Nominal capacity in kW
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01 # 1% loss per hour
    )

In [55]:
network.optimize()

Index(['Battery_bus_2', 'Battery_bus_3', 'Battery_bus_4', 'Battery_bus_5',
       'Battery_bus_6', 'Battery_bus_7', 'Battery_bus_8', 'Battery_bus_9',
       'Battery_bus_10', 'Battery_bus_11', 'Battery_bus_12', 'Battery_bus_13',
       'Battery_bus_14', 'Battery_bus_15', 'Battery_bus_16', 'Battery_bus_17',
       'Battery_bus_18', 'Battery_bus_19', 'Battery_bus_20', 'Battery_bus_21',
       'Battery_bus_22', 'Battery_bus_23', 'Battery_bus_24', 'Battery_bus_25',
       'Battery_bus_26', 'Battery_bus_27', 'Battery_bus_28', 'Battery_bus_29',
       'Battery_bus_30', 'Battery_bus_31', 'Battery_bus_32', 'Battery_bus_33'],
      dtype='str', name='StorageUnit')
Index(['Line_1-2', 'Line_2-3', 'Line_3-4', 'Line_4-5', 'Line_5-6', 'Line_6-7',
       'Line_7-8', 'Line_8-9', 'Line_9-10', 'Line_10-11', 'Line_11-12',
       'Line_12-13', 'Line_13-14', 'Line_14-15', 'Line_15-16', 'Line_16-17',
       'Line_17-18', 'Line_2-19', 'Line_19-20', 'Line_20-21', 'Line_21-22',
       'Line_3-23', 'Line_23-24'

('ok', 'optimal')

In [56]:
print(f"cost: {network.objective:.2f}")


cost: -2706415.57


In [57]:
network.generators_t.p

Generator,Substation,PV_bus_2,PV_bus_3,PV_bus_4,PV_bus_5,PV_bus_6,PV_bus_7,PV_bus_8,PV_bus_9,PV_bus_10,...,PV_bus_24,PV_bus_25,PV_bus_26,PV_bus_27,PV_bus_28,PV_bus_29,PV_bus_30,PV_bus_31,PV_bus_32,PV_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2016-07-15 00:00:00,-1072.457420,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,...,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690
2016-07-15 01:00:00,1036.466106,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,...,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690
2016-07-15 02:00:00,5000.000000,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,...,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690
2016-07-15 03:00:00,5000.000000,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,...,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690,149.394052,149.401885,149.631704,149.106690
2016-07-15 04:00:00,5000.000000,149.396213,149.402134,149.632174,149.109284,149.396213,149.402134,149.632174,149.109284,149.396213,...,149.632174,149.109284,149.396213,149.402134,149.632174,149.109284,149.396213,149.402134,149.632174,149.109284
2016-07-15 05:00:00,-1071.413045,149.408650,149.411643,149.637713,149.124593,149.408650,149.411643,149.637713,149.124593,149.408650,...,149.637713,149.124593,149.408650,149.411643,149.637713,149.124593,149.408650,149.411643,149.637713,149.124593
2016-07-15 06:00:00,-5000.000000,149.429547,149.431489,149.655548,149.163499,149.429547,149.431489,149.655548,149.163499,149.429547,...,149.655548,149.163499,149.429547,149.431489,149.655548,149.163499,149.429547,149.431489,149.655548,149.163499
2016-07-15 07:00:00,-5000.000000,149.481536,149.477577,149.693188,149.244789,149.481536,149.477577,149.693188,149.244789,149.481536,...,149.693188,149.244789,149.481536,149.477577,149.693188,149.244789,149.481536,149.477577,149.693188,149.244789
2016-07-15 08:00:00,-5000.000000,149.569490,149.566526,149.762752,149.394465,149.569490,149.566526,149.762752,149.394465,149.569490,...,149.762752,149.394465,149.569490,149.566526,149.762752,149.394465,149.569490,149.566526,149.762752,149.394465


In [58]:
network.storage_units_t.p

StorageUnit,Battery_bus_2,Battery_bus_3,Battery_bus_4,Battery_bus_5,Battery_bus_6,Battery_bus_7,Battery_bus_8,Battery_bus_9,Battery_bus_10,Battery_bus_11,...,Battery_bus_24,Battery_bus_25,Battery_bus_26,Battery_bus_27,Battery_bus_28,Battery_bus_29,Battery_bus_30,Battery_bus_31,Battery_bus_32,Battery_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2016-07-15 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-07-15 01:00:00,-200.000000,-200.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-200.000000,0.000000,0.000000,-174.606623,0.000000,-151.575893,0.000000,0.000000,0.000000,0.000000
2016-07-15 02:00:00,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-71.883652,0.000000,-200.000000,...,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000
2016-07-15 03:00:00,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,0.000000,-71.568114,-200.000000,-200.000000,...,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000
2016-07-15 04:00:00,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-145.449944,...,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-200.000000,-117.493706,-117.545374,-200.000000
2016-07-15 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-07-15 06:00:00,200.000000,200.000000,200.000000,200.000000,65.500179,63.438959,110.331943,64.834001,200.000000,18.092098,...,200.000000,200.000000,200.000000,200.000000,61.398351,200.000000,200.000000,0.000000,200.000000,0.000000
2016-07-15 07:00:00,200.000000,200.000000,66.845178,200.000000,200.000000,200.000000,0.000000,0.000000,0.000000,0.000000,...,200.000000,0.000000,66.845178,200.000000,0.000000,200.000000,200.000000,200.000000,0.000000,60.784368
2016-07-15 08:00:00,200.000000,17.151110,200.000000,68.176726,200.000000,0.000000,200.000000,0.000000,0.000000,200.000000,...,17.151110,64.156524,200.000000,200.000000,200.000000,0.000000,0.000000,200.000000,0.000000,200.000000
